<a href="https://colab.research.google.com/github/FrancisLa/Langues_de_specialite/blob/main/chaine/03_acquisition/exemple/Moissonnage_exemple_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Moissonnage

**Auteur :** Francis Lareau  
**Date :** 08/10/2026

# 1 Introduction

La collecte de données est l'étape au cours de laquelle on recueille les informations nécessaires à la recherche. Le moissonnage (*scraping* ou *harvesting*) est un moyen parmi d'autres de collecter des données, consistant à explorer des sites web via un script ou un programme afin d'en extraire le contenu. Les pages web renfermant les contenus recherchés sont téléchargées, formant ainsi un ensemble de données brutes que l'on peut ensuite distinguer en trois catégories :

1) données (les textes étudiés) ;  
2) métadonnées (les informations utiles à propos des textes étudiés) ;  
3) informations non retenues (inutiles au regard des objectifs de recherche).

Plusieurs outils sont disponibles pour effectuer le moissonnage du web. Dans le cadre de cette introduction, nous utilisons les bibliothèques **requests**, **urllib.robotparser** et **time**, qui permettent un moissonnage responsable du web, c.-à-d. en suivant les principes de :

1) se présenter à l'hôte ;  
2) demander la permission ;  
3) opérer lentement.

Plus précisément, notre collecte de données impliquera :

1) l'interrogation de sites web pour relever des URL (*Uniform Resource Locator*) pouvant mener à des pages web renfermant les informations recherchées, puis le téléchargement de ces pages web ;  
2) le téléchargement des documents HTML et leur intégration à la structure de données ;  
3) le téléchargement des documents PDF et leur intégration à la structure de données.

Notons que les contenus HTML et PDF téléchargés seront conservés sous forme binaire. La bibliothèque **lxml** sera utilisée pour analyser la structure des documents HTML et en extraire certaines informations.

## 1.1 Librairies

In [17]:
# 1 Charger les bibliothèques
from google.colab import drive

import time
from pathlib import Path
from urllib.parse import urljoin
from urllib.robotparser import RobotFileParser

import pandas as pd
import requests
from lxml import html
import os

### 2.2 Endroit pour le téléchargement

Avant de procéder au moissonnage, connectons-nous à notre Google Drive, puis spécifions le répertoire du projet.

En suite, spécifions le répertoire **download_path** dans lequel seront enregistrées les données. Nous définissons également les répertoires **html_download_path** et **pdf_download_path**, destinés respectivement aux articles au format HTML et PDF.

La classe **Path** de la bibliothèque pathlib permet de représenter et de manipuler des chemins de fichiers et de répertoires. La méthode **Path.cwd()** retourne le répertoire de travail courant. L'attribut parents permet quant à lui d'accéder à ses répertoires parents. Ici, **parents[2]** permet de remonter de trois niveaux dans l'arborescence du projet (puisque le dossier dans lequel se trouve le notebook est trois niveaux sous le répertoire général).

In [ ]:
# 0. Connection à votre Google Drive
drive.mount("/content/drive")

Mounted at /content/drive


In [22]:
# 1. Emplacement racine du projet dans Google Drive.
racine_drive = Path("/content/drive/MyDrive")

# 2. Nom du dossier principal du projet dans Google Drive.
dossier_projet = "Langues_de_specialite"

# 3. Dossier principal du projet (ajouter les dossiers parents si nécessaire).
project_path = racine_drive / "GitHub" / dossier_projet

# 4. Faire du dossier principal le répertoire courant.
os.chdir(project_path)

# 5. Dossier de l'opération en cours
operation_path = project_path / "03_acquisition" / "exemple"

In [20]:
# 6 Noter le chemin d'accès au dossier des données sources ####

download_path = project_path / "donnees"

# 7 Créer le dossier de données brutes s'il n'existe pas ####
download_path.mkdir(parents=True, exist_ok=True)

# 8 Noter le chemin d'accès au dossier des articles au format PDF ####
pdf_download_path = download_path / "sources" / "pdf"

# 9 Créer le dossier du chemin d'accès s'il n'existe pas ####
pdf_download_path.mkdir(parents=True, exist_ok=True)

# 10 Noter le chemin d'accès au dossier des articles au format HTML ####
html_download_path = download_path / "sources" / "html"

# 11 Créer le dossier du chemin d'accès s'il n'existe pas ####
html_download_path.mkdir(parents=True, exist_ok=True)

# 12 Noter un fichier log ####
file_log = download_path / "log.txt"

Note : la méthode mkdir() crée un répertoire. L'argument parents=True crée également les répertoires parents nécessaires, tandis que exist_ok=True évite de produire une erreur si le répertoire existe déjà.

# 2 Collecte des données brutes

## 2.1 Structuration des données brutes

La classe **DataFrame** de la bibliothèque **pandas** fournit une structure de données tabulaire organisée en lignes et en colonnes. Une manière de structurer les contenus extraits du web consiste à insérer une nouvelle ligne dans la structure tabulaire à mesure que de nouveaux documents sont moissonnés, puis à classer les informations relatives à ces documents dans des catégories préalablement identifiées, qui correspondront à des colonnes distinctes.

Le corpus sélectionné pour notre démonstration est la revue en ligne *Argumentation et Analyse du Discours*, de sorte que nos données brutes seront constituées des articles de cette revue. Nous procéderons au moissonnage comme nous le ferions manuellement, en notant d'abord les numéros de la revue, puis les articles de chacun de ces numéros. Une page web au format HTML correspondant à chaque numéro sera téléchargée et l'adresse de cette page sera inscrite dans la colonne **NUM_URL**. Ensuite, une page web au format HTML correspondant à chaque article sera téléchargée et l'adresse de cette page sera inscrite dans la colonne **HTML_URL**. De plus, une version PDF de l'article sera téléchargée et l'adresse de chaque PDF sera inscrite dans la colonne **PDF_URL**. Ultérieurement, les documents HTML et PDF seront intégrés à notre structure de données, respectivement dans les colonnes **HTML** et **PDF**. Nous indiquerons également si l'accès aux documents HTML et PDF a réussi ou non, respectivement dans les colonnes **HTML_ACCESS** et **PDF_ACCESS**, ainsi que la date et l'heure de la tentative d'accès dans les colonnes **HTML_ADATE** et **PDF_ADATE**.

In [ ]:
# Créer un dataframe vide ####
df = pd.DataFrame({
    "ID": pd.Series(dtype="object"),
    "NUM_URL": pd.Series(dtype="object"),
    "HTML_URL": pd.Series(dtype="object"),
    "HTML": pd.Series(dtype="object"),
    "HTML_ACCESS": pd.Series(dtype="object"),
    "HTML_ADATE": pd.Series(dtype="datetime64[ns]"),
    "PDF_URL": pd.Series(dtype="object"),
    "PDF": pd.Series(dtype="object"),
    "PDF_ACCESS": pd.Series(dtype="object"),
    "PDF_ADATE": pd.Series(dtype="datetime64[ns]")
})

## 2.2 Moissonnage

Avant d'élaborer une stratégie de moissonnage, on peut visualiser les opérations que le chercheur effectuerait manuellement pour collecter ses données, en identifiant les opérations répétitives susceptibles d'être automatisées à l'aide de boucles. Par exemple, la revue *Argumentation et Analyse du Discours* possède une page web principale répertoriant les pages web de tous les numéros, lesquelles donnent accès aux pages web des articles de chaque numéro.

La page web principale constituera notre point de départ (https://journals.openedition.org/aad/166) puisque, à partir de celle-ci, il est possible de :

1) noter l'adresse URL de tous les numéros, puis accéder successivement aux pages web de ces numéros ;  
2) noter l'adresse URL des articles de chaque numéro, inscrire cette adresse à la colonne **HTML_URL**, puis accéder et télécharger la page web de ces articles ;  
3) noter l'URL de l'article au format PDF, l'inscrire dans la colonne **PDF_URL**, puis télécharger le PDF.


### 2.2.1 Préparation

Notons que les pages web de la revue commencent par `https://journals.openedition.org`, c'est le nom de domaine. Suivant le nom de domaine, on retrouve `aad` qui est l'acronyme de la revue. Ensemble, celles-ci forment une adresse commune qui est notée dans **main_url**. En Python, la fonction **urljoin()** permet de formuler une adresse URL complète à partir de la partie générale et de la partie particulière.

**requests.Session()** permet de créer un objet **session** que nous utiliserons pour gérer poliment nos transactions avec l'hôte. La propriété **headers** contient les en-têtes HTTP envoyés avec les requêtes. Sa méthode **update()** permet ici d'ajouter l'en-tête **user-agent** afin d'identifier notre script auprès du serveur. Enfin, la classe **RobotFileParser** permet d'interpréter les règles contenues dans un fichier **robots.txt**. La méthode **set_url()** indique l'adresse du fichier **robots.txt** à consulter et la méthode **read()** en effectue la lecture. La méthode **can_fetch()** permet ensuite de vérifier si notre User-Agent est autorisé à accéder à une URL donnée.

In [ ]:
# 1 Noter la partie générale des adresses URL ####
main_url = "https://journals.openedition.org/aad/"

# 2 Coller la partie générale à celle particulière de l'adresse de départ ####
starting_url = urljoin(main_url, "166")

# 3 Ouvrir une session de navigation avec l'hôte ####
session = requests.Session()

# 3.1 Se présenter ####
user_agent = "UMLP's Python Webscraping Tutorial"
session.headers.update({"User-Agent": user_agent})

# 3.2 Lire les permissions indiquées dans robots.txt ####
robots_url = urljoin(main_url, "/robots.txt")
robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()

# 3.3 Vérifier si notre agent est autorisé à consulter la page ####
robots.can_fetch(user_agent, starting_url)

True

### 2.2.2 Moissonnage des numéros

Comme plusieurs étapes du moissonnage nécessiteront d'effectuer les mêmes opérations avant chaque téléchargement, nous les regroupons dans une fonction auxiliaire nommée **polite_get()**. Une fonction permet de regrouper une séquence d'instructions sous un même nom afin de pouvoir la réutiliser. Ici, **polite_get()** vérifie d'abord la permission donnée par **robots.txt**, attend le délai défini, puis envoie la requête HTTP.

La fonction **time.sleep()** suspend l'exécution du programme pendant le nombre de secondes spécifié, ici trois secondes par défaut. La méthode **session.get()** effectue une requête HTTP GET à l'aide de l'objet session défini préalablement afin de récupérer une ressource, tandis que **raise_for_status()** déclenche une erreur lorsque la réponse HTTP indique un échec.

In [ ]:
# Fonction auxiliaire pour effectuer une requête polie ####
def polite_get(url, delay=3):
    # 1 Demander la permission ####
    if not robots.can_fetch(user_agent, url):
        raise PermissionError(f"Le fichier robots.txt interdit le moissonnage de {url}")

    # 2 Opérer lentement ####
    time.sleep(delay)

    # 3 Télécharger la ressource ####
    response = session.get(url, timeout=30)
    response.raise_for_status()
    return response

Dans un premier temps, nous récupérons la page web de départ à l'aide de la fonction **polite_get()**, puis nous extrayons les attributs contenant les URL des numéros. Nous complétons ensuite chacune de ces URL relatives à l'aide de urljoin().

Dans le script ci-dessous :

1) la fonction **polite_get()**, définie précédemment, permet de récupérer la page web de départ ;
2) la fonction **html.fromstring()** transforme le contenu HTML téléchargé en une structure pouvant être parcourue ;
3) la méthode **xpath()** permet de sélectionner certains éléments ou attributs dans cette structure à l'aide d'une expression XPath ;
4) la fonction **urljoin()** permet de convertir une URL relative en URL complète.

In [ ]:
# Noter l'adresse URL de tous les numéros ####

# 1 Naviguer vers la page de départ ####
starting_response = polite_get(starting_url)

# 2 Lire la page web de départ ####
starting_page = html.fromstring(starting_response.content)

# 3 Extraire les attributs où se trouvent les URL ####
numero_hrefs = starting_page.xpath('//*[@id="main"]/ul/li/a/@href')

# 4 Coller la partie générale à celle particulière des URL ####
numero_urls = [urljoin(starting_url, href) for href in numero_hrefs]
numero_urls

['https://journals.openedition.org/aad/10736',
 'https://journals.openedition.org/aad/10000',
 'https://journals.openedition.org/aad/9429',
 'https://journals.openedition.org/aad/9003',
 'https://journals.openedition.org/aad/8427',
 'https://journals.openedition.org/aad/7960',
 'https://journals.openedition.org/aad/7552',
 'https://journals.openedition.org/aad/6996',
 'https://journals.openedition.org/aad/6515',
 'https://journals.openedition.org/aad/5953',
 'https://journals.openedition.org/aad/5424',
 'https://journals.openedition.org/aad/4929',
 'https://journals.openedition.org/aad/4387',
 'https://journals.openedition.org/aad/3877',
 'https://journals.openedition.org/aad/3386',
 'https://journals.openedition.org/aad/2969',
 'https://journals.openedition.org/aad/2568',
 'https://journals.openedition.org/aad/2492',
 'https://journals.openedition.org/aad/2402',
 'https://journals.openedition.org/aad/2301',
 'https://journals.openedition.org/aad/2207',
 'https://journals.openedition.o

Note : les crochets **[]** servent notamment à former des listes en Python. L'expression utilisée ici est une *list comprehension* (compréhension de liste), une syntaxe permettant de construire une nouvelle liste en appliquant une opération à chacun des éléments d'une séquence. Ici, **urljoin()** est appliquée à chaque élément de **numero_hrefs** afin de produire la liste **numero_urls**.

### 2.2.3 Moissonnage des articles

Dans un deuxième temps, on extrait l'URL des articles de chaque numéro et on insère cette information dans le DataFrame à la colonne **HTML_URL**. On ajoute également l'URL du numéro d'où provient cette information à la colonne **NUM_URL**. Plus précisément, l'attribut **shape** indique les dimensions du DataFrame sous la forme (nombre de lignes, nombre de colonnes). Ainsi, **df.shape[0]** retourne le nombre actuel de lignes et fournit ici l'indice de la prochaine ligne à ajouter. L'indexeur **at[]** permet de lire ou de modifier une valeur unique du DataFrame à partir de son indice et du nom de sa colonne.  Ici, il sert à inscrire une valeur dans une cellule précise du DataFrame en indiquant l'indice de la ligne et le nom de la colonne.

In [ ]:
for numero_url in numero_urls:

    # 1 Naviguer vers la page du numéro ####
    numero_response = polite_get(numero_url)
    numero_page = html.fromstring(numero_response.content)

    # 2 Noter l'adresse URL de tous les articles ####
    article_hrefs = numero_page.xpath('//div[@class="title"]/a/@href')

    # 3 Noter l'URL de chaque article dans le dataframe
    for article_href in article_hrefs:
        i = df.shape[0] # identifiant de la prochaine ligne du dataframe
        df.at[i,"ID"] = "aad" + article_href
        df.at[i,"NUM_URL"] = numero_url
        df.at[i, "HTML_URL"] = urljoin(starting_url, article_href)

df.head()

,ID,NUM_URL,HTML_URL,HTML,HTML_ACCESS,HTML_ADATE,PDF_URL,PDF,PDF_ACCESS,PDF_ADATE
0,aad11348,https://journals.openedition.org/aad/10736,https://journals.openedition.org/aad/11348,NaN,NaN,NaT,NaN,NaN,NaN,NaT
1,aad10930,https://journals.openedition.org/aad/10736,https://journals.openedition.org/aad/10930,NaN,NaN,NaT,NaN,NaN,NaN,NaT
2,aad10742,https://journals.openedition.org/aad/10736,https://journals.openedition.org/aad/10742,NaN,NaN,NaT,NaN,NaN,NaN,NaT
3,aad10812,https://journals.openedition.org/aad/10736,https://journals.openedition.org/aad/10812,NaN,NaN,NaT,NaN,NaN,NaN,NaT
4,aad10860,https://journals.openedition.org/aad/10736,https://journals.openedition.org/aad/10860,NaN,NaN,NaT,NaN,NaN,NaN,NaT


Il est préférable de sauvegarder souvent tout au long du moissonnage de manière à minimiser le nombre de requêtes. La méthode **to_pickle()** de pandas sérialise un DataFrame afin de l'enregistrer dans un fichier pickle. La fonction **pd.read_pickle()** permet ensuite de recharger ce fichier et de reconstruire le DataFrame.

In [ ]:
# Sauvegarder le dataframe au format pickle ####
df.to_pickle(download_path / "AAD_dataframe_full_inventory.pkl")

Pour cette démonstration, nous moissonnerons seulement le premier numéro: https://journals.openedition.org/aad/171. L'expression **df["NUM_URL"] == "https://journals.openedition.org/aad/171"** produit une série de valeurs booléennes (True ou False) indiquant, pour chaque ligne, si la valeur de **NUM_URL** correspond à l'URL recherchée. Cette série est ensuite utilisée pour sélectionner uniquement les lignes correspondantes. Enfin, la méthode **copy()** crée une copie indépendante de ce sous-ensemble afin de former un nouveau DataFrame.

In [ ]:
df = pd.read_pickle(download_path / "AAD_dataframe_full_inventory.pkl")
df = df[df["NUM_URL"] == "https://journals.openedition.org/aad/171"].copy()
df.head()

,ID,NUM_URL,HTML_URL,HTML,HTML_ACCESS,HTML_ADATE,PDF_URL,PDF,PDF_ACCESS,PDF_ADATE
443,aad184,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/184,NaN,NaN,NaT,NaN,NaN,NaN,NaT
444,aad193,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/193,NaN,NaN,NaT,NaN,NaN,NaN,NaT
445,aad198,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/198,NaN,NaN,NaT,NaN,NaN,NaN,NaT
446,aad263,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/263,NaN,NaN,NaT,NaN,NaN,NaN,NaT
447,aad251,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/251,NaN,NaN,NaT,NaN,NaN,NaN,NaT


### 2.2.4 Téléchargement des documents HTML

Procédons maintenant au téléchargement des pages web HTML de chaque article de ce numéro :

1) Pour chaque ligne du DataFrame, la méthode **iterrows()** parcourt le DataFrame ligne par ligne et fournit, pour chaque ligne, son indice (**index**) et son contenu (**row**).
2) La fonction **pd.isna()** permet de vérifier si une valeur est manquante. La fonction **str()** convertit une valeur en chaîne de caractères et la méthode **strip()** en retire les espaces au début et à la fin. L'instruction **continue** permet alors de passer directement à la ligne suivante du DataFrame lorsque l'URL est absente ou vide.
3) La structure **try/except** permet de gérer les erreurs sans interrompre l'ensemble du moissonnage. Python tente d'abord d'exécuter les instructions placées dans le bloc **try**. Si une erreur survient, le bloc **except** est exécuté : l'échec est alors inscrit dans le DataFrame et dans le fichier journal, puis la boucle peut poursuivre le traitement des autres articles.
4) La méthode **write_bytes()** de **Path** enregistre directement un contenu binaire dans un fichier.
5) **pd.Timestamp.now()** retourne la date et l'heure au moment de l'exécution.
6) La méthode **open()** ouvre ici le fichier journal (log) en mode **"a"** (append), ce qui ajoute les nouvelles informations à la fin du fichier sans supprimer son contenu précédent au moyen de la méthode **write()** qui inscrit une chaîne de caractères dans le fichier ouvert. L'instruction **with** au début assure sa fermeture automatique une fois l'écriture terminée.

In [ ]:
# Boucle pour télécharger les articles au format HTML ####

for index, row in df.iterrows():

    aad_identifier = row["ID"]

    article_url = row["HTML_URL"]

    # Ignorer les lignes sans URL HTML
    if pd.isna(article_url) or not str(article_url).strip():
        continue

    try:

        # Télécharger l'article HTML ####
        article_response = polite_get(article_url)

        # Déterminer le nom du fichier ####
        html_destfile = f"{aad_identifier}.html"

        # Déterminer le chemin du fichier ####
        html_file = html_download_path / html_destfile

        # Enregistrer le fichier ####
        html_file.write_bytes(article_response.content)

        # Inscrire le HTML dans le dataframe ####
        df.at[index, "HTML"] = article_response.content

        # Inscrire le statut de l'accès et la date ####
        time_now = pd.Timestamp.now()
        df.at[index, "HTML_ACCESS"] = True
        df.at[index, "HTML_ADATE"] = time_now

        # Ajouter une entrée au log ####
        with file_log.open("a", encoding="utf-8") as log:
            log.write(
                f"{time_now} - HTML : {article_url}\n"
            )

    except Exception as error:

        time_now = pd.Timestamp.now()
        df.at[index, "HTML_ACCESS"] = False
        df.at[index, "HTML_ADATE"] = time_now

        with file_log.open("a", encoding="utf-8") as log:
            log.write(
                f"{time_now} - ERREUR HTML : "
                f"{article_url} - {error}\n"
            )

df.head()

,ID,NUM_URL,HTML_URL,HTML,HTML_ACCESS,HTML_ADATE,PDF_URL,PDF,PDF_ACCESS,PDF_ADATE
443,aad184,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/184,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:24.180008,NaN,NaN,NaN,NaT
444,aad193,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/193,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:27.525340,NaN,NaN,NaN,NaT
445,aad198,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/198,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:30.866647,NaN,NaN,NaN,NaT
446,aad263,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/263,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:34.179310,NaN,NaN,NaN,NaT
447,aad251,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/251,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:37.425106,NaN,NaN,NaN,NaT


Note : la fonction **str()** convertit une valeur en chaîne de caractères, tandis que la méthode **strip()** retire les espaces situés au début et à la fin de cette chaîne. Ces opérations permettent ici de détecter également une URL vide ou composée uniquement d'espaces. Enfin, l'instruction **continue** permet de passer directement à l'itération suivante de la boucle lorsque l'URL est absente.

### 2.2.4 Téléchargement des documents PDF

Procédons au téléchargement du PDF de chaque article. Après avoir analysé le HTML avec **html.fromstring()**, nous utilisons de nouveau **xpath()** pour identifier le lien permettant de télécharger le PDF. La méthode **headers.get()** permet ensuite de consulter l'en-tête HTTP Content-Type, tandis que **startswith()** vérifie si le contenu téléchargé commence par la signature **%PDF**, caractéristique d'un fichier PDF.

In [ ]:
# Boucle pour télécharger les articles au format PDF ####
for index, row in df.iterrows():

    aad_identifier = row["ID"]
    article_url = row["HTML_URL"]
    html_content = row["HTML"]

    # Ignorer les lignes sans contenu HTML
    if html_content is None or (
        isinstance(html_content, float) and pd.isna(html_content)
    ):
        continue

    try:

        # Lire la page HTML téléchargée ####
        html_tree = html.fromstring(html_content)

        # Extraire le href du bouton de téléchargement ####
        pdf_links = html_tree.xpath('//*[@id="wDownload"]/@href')

        if not pdf_links:
            raise ValueError(
                'Aucun lien trouvé avec le XPath //*[@id="wDownload"]/@href'
            )

        # Convertir le lien relatif en URL absolue ####
        pdf_url = urljoin(article_url, pdf_links[0])

        # Inscrire l'URL du PDF dans le dataframe ####
        df.at[index, "PDF_URL"] = pdf_url

        # Télécharger le PDF ####
        pdf_response = polite_get(pdf_url)

        # Vérifier que le contenu est bien un PDF ####
        content_type = pdf_response.headers.get(
            "Content-Type",
            ""
        ).lower()

        if not pdf_response.content:
            raise ValueError("Le PDF téléchargé est vide")

        if (
            not pdf_response.content.startswith(b"%PDF")
            and "application/pdf" not in content_type
        ):
            raise ValueError(
                f"Le contenu téléchargé n'est pas un PDF : {content_type}"
            )

        # Déterminer le nom et le chemin du fichier ####
        pdf_destfile = f"{aad_identifier}.pdf"
        pdf_file = pdf_download_path / pdf_destfile

        # Enregistrer le PDF ####
        pdf_file.write_bytes(pdf_response.content)

        # Inscrire le PDF dans le dataframe ####
        df.at[index, "PDF"] = pdf_response.content

        # Inscrire le statut de l'accès et la date ####
        time_now = pd.Timestamp.now()

        df.at[index, "PDF_ACCESS"] = True
        df.at[index, "PDF_ADATE"] = time_now

        # Ajouter une entrée au log ####
        with file_log.open("a", encoding="utf-8") as log:
            log.write(
                f"{time_now} - PDF : {pdf_url}\n"
            )

    except Exception as error:

        time_now = pd.Timestamp.now()

        df.at[index, "PDF_ACCESS"] = False
        df.at[index, "PDF_ADATE"] = time_now

        with file_log.open("a", encoding="utf-8") as log:
            log.write(
                f"{time_now} - ERREUR PDF : "
                f"{article_url} - {error}\n"
            )

df.head()

,ID,NUM_URL,HTML_URL,HTML,HTML_ACCESS,HTML_ADATE,PDF_URL,PDF,PDF_ACCESS,PDF_ADATE
443,aad184,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/184,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:24.180008,https://journals.openedition.org/aad/pdf/184,b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n4 0 obj\n\n<<\n...,True,2026-10-30 19:06:32.060454
444,aad193,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/193,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:27.525340,https://journals.openedition.org/aad/pdf/193,b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n4 0 obj\n\n<<\n...,True,2026-10-30 19:06:36.173508
445,aad198,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/198,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:30.866647,https://journals.openedition.org/aad/pdf/198,b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n4 0 obj\n\n<<\n...,True,2026-10-30 19:06:40.052876
446,aad263,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/263,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:34.179310,https://journals.openedition.org/aad/pdf/263,b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n4 0 obj\n\n<<\n...,True,2026-10-30 19:06:44.155059
447,aad251,https://journals.openedition.org/aad/171,https://journals.openedition.org/aad/251,"b'<!DOCTYPE html PUBLIC ""-//W3C//DTD XHTML 1.0...",True,2026-10-30 19:05:37.425106,https://journals.openedition.org/aad/pdf/251,b'%PDF-1.7\n%\xe2\xe3\xcf\xd3\n4 0 obj\n\n<<\n...,True,2026-10-30 19:06:48.685036


# 3 Sauvegarde de la structure de données

Nous produisons deux types de sauvegardes. D'abord, **to_pickle()** permet de conserver le DataFrame complet, y compris les contenus HTML et PDF sous forme binaire. Ensuite, nous créons avec **drop()** une version du DataFrame dans laquelle les colonnes HTML et PDF sont retirées. Cet inventaire, plus léger et facilement consultable, est enregistré avec **to_pickle**, mais aussi **to_excel()** au format Excel et **to_csv()** au format CSV. L'argument **index=False** évite d'ajouter l'indice du DataFrame comme colonne supplémentaire.

In [ ]:
# Sauvegarder le dataframe au format pickle ####
df.to_pickle(
    download_path / "sources" / "AAD_dataframe_partial_inventory_with_data.pkl"
)

# Faire un dataframe sans les colones HTML et PDF pour générer l'inventaire
df_inventory = df.drop(columns=["HTML", "PDF"])

# Sauvegarder le dataframe au format pickle ####
df_inventory.to_pickle(
    download_path / "AAD_dataframe_partial_inventory.pkl"
)

# Sauvegarder le dataframe au format .xlsx ####
df_inventory.to_excel(
    download_path / "AAD_dataframe_partial_inventory.xlsx",
    index=False
)

# Sauvegarder le dataframe au format .csv ####
df_inventory.to_csv(
    download_path / "AAD_dataframe_partial_inventory.csv",
    index=False,
    encoding="utf-8"
)

# 4 Exercices

L'objectif de la série d'exercices est de moissonner la *Revue d'Anthropologie des Connaissances* disponible en ligne sur OpenEdition (`https://journals.openedition.org/rac/263`). Pour ce faire, nous devons identifier des URL à l'aide de requêtes XPath. Notons que ces adresses ou hyperliens correspondent généralement à des éléments dont l'attribut **href** spécifie l'URL. Directement dans le navigateur, on peut inspecter un hyperlien en faisant un clic droit sur l'élément qui nous intéresse, puis en cliquant sur *Inspecter*.

### 4.1 Préparation

Débutons par l'ouverture d'une session avec **requests.Session()**. Exécutez le code suivant après avoir déterminé la partie générale et la partie particulière de la page de départ.

In [ ]:
# 1 Noter la partie générale des adresses URL ####
main_url = "..."  # Veuillez remplacer ... par la partie générale de l'URL ####

# 2 Coller la partie générale à celle particulière de l'adresse de départ ####
starting_url = urljoin(main_url, "...")  # Veuillez remplacer ... par la partie particulière de l'URL ####

# 3 Ouvrir une session de navigation avec l'hôte ####
session = requests.Session()

# 3.1 Se présenter ####
user_agent = #Veillez décrire votre robot
session.headers.update({"User-Agent": user_agent})
delay = 1

# 3.2 Lire les permissions indiquées dans robots.txt ####
robots_url = urljoin(main_url, "/robots.txt")
robots = RobotFileParser()
robots.set_url(robots_url)
robots.read()

# 3.3 Vérifier si notre agent est autorisé à consulter la page ####
robots.can_fetch(user_agent, starting_url)

### 4.2 URL des numéros

Nous allons extraire l'URL de tous les numéros présents sur la page web de départ contenant ces informations. Veuillez compléter le script suivant en inscrivant la formule XPath.

In [ ]:
numero_url = "https://journals.openedition.org/rac/2162"

numero_response = polite_get(numero_url)
numero_page = html.fromstring(numero_response.content)

article_hrefs = numero_page.xpath("...")  # Veuillez remplacer ... par la formule xpath ####
article_urls = [urljoin(numero_url, href) for href in article_hrefs]

## 4.3 Le moissonnage d'un numéro de la *Revue d'Anthropologie des Connaissances*

### 4.3.1 Moissonnage

Inscrivez les URL de tous les articles du numéro précédent (`https://journals.openedition.org/rac/2162`) dans des cellules distinctes d'une structure de données **df**.

In [ ]:
# 1 Créer un dataframe vide ####
df = pd.DataFrame({
    "ID": pd.Series(dtype="object"),
    "NUM_URL": pd.Series(dtype="object"),
    "HTML_URL": pd.Series(dtype="object"),
    "HTML": pd.Series(dtype="object"),
    "HTML_ACCESS": pd.Series(dtype="object"),
    "HTML_ADATE": pd.Series(dtype="datetime64[ns]"),
    "PDF_URL": pd.Series(dtype="object"),
    "PDF": pd.Series(dtype="object"),
    "PDF_ACCESS": pd.Series(dtype="object"),
    "PDF_ADATE": pd.Series(dtype="datetime64[ns]")
})

# 2 Noter le chemin d'accès au dossier des articles au format PDF ####
pdf_download_path = download_path / "exemples" / "pdf"

# 3 Créer le dossier du chemin d'accès s'il n'existe pas ####
pdf_download_path.mkdir(parents=True, exist_ok=True)

# 4 Noter le chemin d'accès au dossier des articles au format HTML ####
html_download_path = download_path / "exemples" / "html"

# 5 Créer le dossier du chemin d'accès s'il n'existe pas ####
html_download_path.mkdir(parents=True, exist_ok=True)

# 6 Noter l'URL de chaque article dans le dataframe
for article_href in article_hrefs:

    # Veuillez compléter le code permettant d'ajouter la ligne à df ####

    # Veuillez compléter le code permettant de peupler les colonnes ID, NUM_URL et HTML_URL ####


### 4.3.2 Importation des documents HTML

En vous inspirant du script de la section [2.2.4 Téléchargement des documents HTML](#224-téléchargement-des-documents-html), veuillez inscrire les documents HTML dans la colonne **HTML** de la structure de données **df**. De plus, veillez télécharger les documents HTML dans **html_download_path** en leur donnant un nom approprié.

In [ ]:
# Veuillez inscrire le code ####

## 4.4 Sauvegarde de la structure de données

Veuillez enregistrer la structure de données résultante **df** dans un dossier cohérent au moyen d'un fichier pickle muni d'un nom approprié.

In [ ]:
# Veuillez inscrire le code ####